# Module 1 — Writing Production-Quality Python
## From scripts that work to code you can trust

This module assumes you know basic Python syntax. The goal is to write code that is easier to test, debug, reuse, and maintain in data science and AI projects.

By the end, you should be able to:
- design functions with clear inputs and outputs;
- understand mutability and avoid common Python surprises;
- use comprehensions, iterators, and generators appropriately;
- handle errors intentionally;
- add type hints and docstrings;
- write simple tests;
- separate reusable logic from notebook-only experimentation.

## 1. Functions: one job, clear contract

A good function usually has one clear responsibility. It should make its inputs, outputs, and failure behavior understandable.

Avoid functions that silently depend on variables created elsewhere in a notebook. Hidden state makes code harder to test and reuse.

In [ ]:
def calculate_pmpm(total_paid: float, member_months: int) -> float:
    """Calculate paid-per-member-per-month.

    Args:
        total_paid: Total paid amount for the period.
        member_months: Total member-months represented by the data.

    Returns:
        Cost per member per month.

    Raises:
        ValueError: If member_months is not positive.
    """
    if member_months <= 0:
        raise ValueError("member_months must be greater than zero")

    return total_paid / member_months


assert calculate_pmpm(120_000, 1_000) == 120.0

### Exercise 1
Write a function `safe_percentage(part, total)` that returns `part / total * 100`. Raise `ValueError` if `total` is zero or negative. Try it before looking at the solution.

In [ ]:
def safe_percentage(part: float, total: float) -> float:
    if total <= 0:
        raise ValueError("total must be positive")
    return (part / total) * 100


assert safe_percentage(25, 100) == 25.0
try:
    safe_percentage(1, 0)
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for a zero denominator")

## 2. Mutable objects and the default-argument trap

Lists and dictionaries are mutable: their contents can change after creation. A default argument is evaluated once when the function is defined, not each time it is called.

**Avoid:** `def add_item(item, items=[]): ...`

Use `None` as the default and create a new list inside the function.

In [ ]:
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items


first = add_item("apple")
second = add_item("banana")
assert first == ["apple"]
assert second == ["banana"]

### Exercise 2
Explain why a mutable default can accidentally share data across function calls. Then write a safe function that appends a value to a provided list, or creates a new list if none is provided.

## 3. Copying, aliasing, and mutation

Two variable names can refer to the same list. Changing the list through one name is visible through the other. A shallow copy creates a new outer list but does not recursively copy nested objects.

In [ ]:
original = [[1, 2], [3, 4]]
alias = original
shallow_copy = original.copy()

alias[0].append(99)
assert original[0] == [1, 2, 99]  # alias and original are the same list
assert shallow_copy[0] == [1, 2, 99]  # nested list is still shared

# Deep copy recursively copies nested objects.
import copy
deep_copy = copy.deepcopy(original)
original[0].append(100)
assert deep_copy[0] == [1, 2, 99]

**Data science connection:** When transforming a DataFrame or nested configuration, understand whether an operation mutates an object or creates a new one. Accidental mutation can make notebook results depend on cell execution order.

## 4. Comprehensions: concise, not cryptic

Use comprehensions for simple transformations and filters. If the logic becomes difficult to read, use a normal loop or a helper function.

In [ ]:
values = [3, -2, 0, 7, -5]

squares_of_positive = [value ** 2 for value in values if value > 0]
value_to_sign = {
    value: ("positive" if value > 0 else "negative" if value < 0 else "zero")
    for value in values
}

assert squares_of_positive == [9, 49]
assert value_to_sign[0] == "zero"

### Exercise 3
Given `scores = [55, 82, 91, 47, 76]`, create a list of scores at least 70 and a dictionary mapping each score to `"pass"` if it is at least 60, otherwise `"review"`.

In [ ]:
scores = [55, 82, 91, 47, 76]
high_scores = [score for score in scores if score >= 70]
status = {score: ("pass" if score >= 60 else "review") for score in scores}

assert high_scores == [82, 91, 76]
assert status[55] == "review"
assert status[82] == "pass"

## 5. Iterables, iterators, and generators

An **iterable** can produce an iterator. An **iterator** produces one item at a time. A **generator** is a convenient way to create an iterator using `yield`.

Generators are useful when processing large files or streams because they do not need to hold every output in memory at once.

In [ ]:
def read_in_batches(values, batch_size):
    """Yield successive batches without building all batches at once."""
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    batch = []
    for value in values:
        batch.append(value)
        if len(batch) == batch_size:
            yield batch
            batch = []

    if batch:
        yield batch


assert list(read_in_batches([1, 2, 3, 4, 5], 2)) == [[1, 2], [3, 4], [5]]

### Exercise 4
Write a generator that yields only positive values from an iterable. Explain why this can be more memory-efficient than creating a second full list.

In [ ]:
def positive_values(values):
    for value in values:
        if value > 0:
            yield value


assert list(positive_values([-2, 0, 3, 8, -1])) == [3, 8]

## 6. Exceptions: fail clearly and narrowly

Catch an exception only when you can handle it or add useful context. Avoid broad `except Exception` blocks that hide programming errors.

For data pipelines, you may choose to log a malformed row and continue, or fail the job. The right choice depends on whether that data issue is recoverable.

In [ ]:
def parse_integer(value):
    try:
        return int(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Expected an integer-like value, received {value!r}") from exc


assert parse_integer("42") == 42
try:
    parse_integer("not-a-number")
except ValueError as exc:
    assert "Expected an integer-like value" in str(exc)

## 7. Type hints and docstrings

Type hints communicate intended inputs and outputs. They do not automatically validate data at runtime, but they help editors, reviewers, and static-analysis tools.

Prefer clear names and documented assumptions over overly complex annotations.

In [ ]:
from typing import Sequence


def mean_value(values: Sequence[float]) -> float:
    """Return the arithmetic mean of a non-empty sequence."""
    if len(values) == 0:
        raise ValueError("values must not be empty")
    return sum(values) / len(values)


assert mean_value([2.0, 4.0, 6.0]) == 4.0

## 8. Tests and assertions

Assertions are useful for quick learning checks and internal invariants. For project code, use a test framework such as `pytest` so tests can be discovered and run repeatedly.

Test:
- a normal input;
- the smallest valid input;
- empty input where allowed;
- invalid input;
- duplicate values or missing values where relevant.

In [ ]:
def normalize_name(name: str) -> str:
    """Strip outer whitespace and collapse repeated internal spaces."""
    return " ".join(name.split())


def test_normalize_name():
    assert normalize_name("  Ada   Lovelace ") == "Ada Lovelace"
    assert normalize_name("") == ""
    assert normalize_name("Grace") == "Grace"


test_normalize_name()

## 9. Logging instead of scattered print statements

`print()` is fine for quick exploration. In reusable scripts and pipelines, logging gives you levels, timestamps, and a consistent way to report progress or failures.

In [ ]:
import logging

logger = logging.getLogger("data_pipeline")
logger.setLevel(logging.INFO)

# Add a handler only if the application has not configured one already.
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())

def validate_nonnegative(values):
    invalid_count = sum(value < 0 for value in values)
    if invalid_count:
        logger.warning("Found %d negative values", invalid_count)
    return invalid_count == 0


assert validate_nonnegative([0, 1, 2]) is True
assert validate_nonnegative([0, -1]) is False

## 10. Organize notebook code into reusable modules

A notebook is excellent for exploration, but reusable functions should eventually live in `.py` modules. A practical structure might be:

```text
project/
├── notebooks/
│   └── exploration.ipynb
├── src/
│   └── project_name/
│       ├── __init__.py
│       ├── cleaning.py
│       └── metrics.py
├── tests/
│   └── test_metrics.py
├── requirements.txt
└── README.md
```

Keep data exploration, reusable business logic, tests, and configuration separate. Avoid hard-coded secrets and machine-specific file paths.

# Practice Assignment — Build a Reusable Data-Quality Helper

Implement a function that:
1. receives a list of numeric values;
2. returns the count of missing values represented by `None`;
3. returns the count of negative numeric values;
4. does not modify the input list;
5. raises `TypeError` if a non-numeric, non-`None` value is encountered.

Try it yourself before running the reference implementation below.

In [ ]:
from numbers import Real


def summarize_numeric_values(values):
    missing_count = 0
    negative_count = 0

    for value in values:
        if value is None:
            missing_count += 1
        elif isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError(f"Expected a number or None, got {type(value).__name__}")
        elif value < 0:
            negative_count += 1

    return {
        "missing_count": missing_count,
        "negative_count": negative_count,
    }


sample = [10, None, -2, 0, None, 3.5]
sample_before = sample.copy()
assert summarize_numeric_values(sample) == {"missing_count": 2, "negative_count": 1}
assert sample == sample_before  # Function must not mutate the input.

try:
    summarize_numeric_values([1, "bad", None])
except TypeError:
    pass
else:
    raise AssertionError("Expected TypeError for a string value")

# Module 1 — Readiness Check

You are ready to move on when you can explain and demonstrate:
- [ ] Why mutable default arguments can be dangerous.
- [ ] The difference between aliasing, shallow copy, and deep copy.
- [ ] When a generator is useful.
- [ ] Why narrow exception handling is safer.
- [ ] What type hints do and do not guarantee.
- [ ] How to test normal, boundary, and invalid inputs.
- [ ] Why reusable code should not depend on notebook execution order.

**Next module:** NumPy and Pandas for efficient, memory-aware data processing.